# ETL 04 — Limpeza, tipagem e padronização das bases

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Entrada:** arquivos brutos de `data/raw/` gerados pelos notebooks 01 (BACEN), 02 (B3) e 03 (CVM).

**Tratamentos:** conversão de datas com formato explícito, tipagem numérica, remoção de duplicados pela chave de negócio, tratamento de nulos, padronização dos nomes de colunas e conversão de escala monetária.

**Saída:** tabelas tratadas em `data/processed/` (versionadas — servem de contingência para a aplicação).

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

qualidade = []  # acumula o diagnóstico de cada base, antes e depois do tratamento


def diagnostico(df, base, chaves):
    """Resumo de qualidade: total de linhas, valores nulos e duplicados pela chave de negócio."""
    return {
        "base": base,
        "linhas": len(df),
        "nulos": int(df.isna().sum().sum()),
        "duplicados_na_chave": int(df.duplicated(subset=chaves).sum()),
    }

In [2]:
SERIES_SGS = {432: "selic_meta", 433: "ipca_mensal", 13522: "ipca_12m", 1: "ptax_venda"}

frames = []
for codigo, nome in SERIES_SGS.items():
    df = pd.read_csv(RAW_DIR / f"bacen_{codigo}_{nome}.csv")
    df["serie"] = nome
    frames.append(df)
macro_bruto = pd.concat(frames, ignore_index=True)

macro = macro_bruto.copy()
macro["data"] = pd.to_datetime(macro["data"], format="%d/%m/%Y")  # formato brasileiro explícito
macro["valor"] = pd.to_numeric(macro["valor"], errors="coerce")
macro = macro.drop_duplicates(subset=["serie", "data"]).dropna(subset=["valor"])
macro = macro[["data", "serie", "valor"]].sort_values(["serie", "data"]).reset_index(drop=True)

macro.to_csv(PROC_DIR / "fato_macro.csv", index=False)

qualidade.append(diagnostico(macro_bruto, "BACEN — bruto", ["serie", "data"]))
qualidade.append(diagnostico(macro, "BACEN — tratado", ["serie", "data"]))

# Prova de que as datas foram lidas no padrão brasileiro
macro.groupby("serie")["data"].agg(["min", "max", "count"])

,min,max,count
serie,,,
ipca_12m,2019-01-01,2023-12-01,60
ipca_mensal,2019-01-01,2023-12-01,60
ptax_venda,2019-01-02,2023-12-29,1255
selic_meta,2019-01-01,2023-12-31,1826


In [3]:
cot_bruto = pd.read_csv(RAW_DIR / "b3_cotacoes_2019_2023.csv", parse_dates=["Date"])

cot = cot_bruto.rename(columns={
    "Date": "data", "Open": "abertura", "High": "maxima", "Low": "minima",
    "Close": "fechamento", "Adj Close": "fechamento_ajustado", "Volume": "volume_acoes",
})
cot = cot.drop_duplicates(subset=["ticker", "data"]).dropna(subset=["fechamento_ajustado"])

# Volume financeiro estimado: preço de fechamento (nominal) × quantidade de ações negociadas
cot["volume_financeiro"] = cot["fechamento"] * cot["volume_acoes"]

colunas = ["ticker", "data", "abertura", "maxima", "minima", "fechamento",
           "fechamento_ajustado", "volume_acoes", "volume_financeiro"]
cot = cot[colunas].sort_values(["ticker", "data"]).reset_index(drop=True)

cot.to_csv(PROC_DIR / "fato_cotacoes.csv", index=False)

qualidade.append(diagnostico(cot_bruto, "B3 — bruto", ["ticker", "Date"]))
qualidade.append(diagnostico(cot, "B3 — tratado", ["ticker", "data"]))
cot.head()

,ticker,data,abertura,maxima,minima,fechamento,fechamento_ajustado,volume_acoes,volume_financeiro
0,ABEV3,2019-01-02,15.400000,16.299999,15.400000,16.150000,11.603578,18692900,3.018903e+08
1,ABEV3,2019-01-03,16.010000,16.330000,15.890000,16.330000,11.732904,22739200,3.713311e+08
2,ABEV3,2019-01-04,16.299999,16.690001,16.030001,16.549999,11.890971,27484600,4.548701e+08
3,ABEV3,2019-01-07,16.570000,16.629999,16.340000,16.480000,11.840679,17084500,2.815526e+08
4,ABEV3,2019-01-08,16.540001,16.700001,16.090000,16.150000,11.603578,22108700,3.570555e+08


In [4]:
dfp_bruto = pd.read_csv(RAW_DIR / "cvm_dfp_carteira_2019_2023.csv",
                        dtype={"CNPJ_CIA": str, "CD_CONTA": str})

dfp = dfp_bruto.rename(columns={
    "TICKER": "ticker", "CNPJ_CIA": "cnpj", "DENOM_CIA": "empresa",
    "DEMONSTRATIVO": "demonstrativo", "DT_FIM_EXERC": "data_fim_exercicio",
    "CD_CONTA": "cd_conta", "DS_CONTA": "ds_conta", "VL_CONTA": "valor_mil",
})
dfp["data_fim_exercicio"] = pd.to_datetime(dfp["data_fim_exercicio"], format="%Y-%m-%d")
dfp["ano"] = dfp["data_fim_exercicio"].dt.year
dfp["valor_mil"] = pd.to_numeric(dfp["valor_mil"], errors="coerce")

colunas = ["ticker", "cnpj", "empresa", "ano", "demonstrativo", "cd_conta", "ds_conta", "valor_mil"]
chave = ["ticker", "ano", "demonstrativo", "cd_conta"]
qualidade.append(diagnostico(dfp[colunas], "CVM — bruto", chave))

dfp["valor_reais"] = dfp["valor_mil"] * 1000  # ESCALA_MOEDA = MIL → converte para reais
dfp = dfp.drop_duplicates(subset=chave).dropna(subset=["valor_reais"])
dfp = (dfp[["ticker", "cnpj", "empresa", "ano", "demonstrativo", "cd_conta", "ds_conta", "valor_reais"]]
       .sort_values(["ticker", "ano", "demonstrativo", "cd_conta"]).reset_index(drop=True))

dfp.to_csv(PROC_DIR / "fato_contabil.csv", index=False)

qualidade.append(diagnostico(dfp, "CVM — tratado", chave))
dfp.groupby(["ticker", "demonstrativo"])["ano"].nunique().unstack()

demonstrativo,BPA,BPP,DRE
ticker,,,
ABEV3,5,5,5
EGIE3,5,5,5
EMBR3,5,5,5
ITUB4,5,5,5
LREN3,5,5,5
PETR4,5,5,5
SUZB3,5,5,5
VALE3,5,5,5
WEGE3,5,5,5


In [5]:
for arquivo in sorted(PROC_DIR.glob("*.csv")):
    print(f"{arquivo.name}: {arquivo.stat().st_size / 1e6:.2f} MB")

pd.DataFrame(qualidade)

fato_contabil.csv: 1.01 MB
fato_cotacoes.csv: 1.49 MB
fato_macro.csv: 0.09 MB


,base,linhas,nulos,duplicados_na_chave
0,BACEN — bruto,3201,0,0
1,BACEN — tratado,3201,0,0
2,B3 — bruto,11169,0,0
3,B3 — tratado,11169,0,0
4,CVM — bruto,9894,0,0
5,CVM — tratado,9894,0,0


## Conclusão do tratamento

| Base | Linhas | Nulos | Duplicados | Saída |
|---|---|---|---|---|
| BACEN | 3.201 (1.826 + 60 + 60 + 1.255) | 0 | 0 | `fato_macro.csv` |
| B3 | 11.169 (9 × 1.241) | 0 | 0 | `fato_cotacoes.csv` |
| CVM | 9.894 (9 empresas × 5 anos × 3 demonstrativos) | 0 | 0 | `fato_contabil.csv` |

**Leitura crítica:** as fontes oficiais chegaram íntegras — nenhuma linha precisou ser removida. As verificações de nulos e duplicados funcionam aqui como **validação** (garantem a integridade em vez de presumi-la) e como **proteção** para execuções futuras.

**Transformações efetivamente aplicadas:**
- Datas em texto convertidas com formato explícito: `%d/%m/%Y` (BACEN) e `%Y-%m-%d` (CVM), evitando a inversão dia/mês;
- Valores contábeis convertidos de milhares para reais (`ESCALA_MOEDA = MIL` × 1.000);
- Colunas padronizadas em português, minúsculas e sem espaços (compatíveis com SQL);
- Cálculo do volume financeiro estimado (fechamento nominal × quantidade negociada);
- Ano do exercício extraído da data de encerramento (CVM).